# Statistics for Data Analysts/Engineers — Module 13: Logistic Regression

The linear regression from Module 12 predicts a CONTINUOUS variable. But very often
we care about a BINARY outcome: did a customer convert (yes/no), is a transaction
fraudulent (yes/no), will a customer churn (yes/no). **Logistic regression** is the
standard tool for exactly this kind of task -- we'll use our `converted` column.

## Table of contents
1. [Why not plain linear regression?](#sec1)
2. [The logistic function and the logit](#sec2)
3. [Fitting a logistic regression](#sec3)
4. [Interpretation: the odds ratio](#sec4)
5. [Predicting probabilities](#sec5)
6. [Decision threshold and confusion matrix](#sec6)
7. [Precision and recall](#sec7)
8. [Summary and exercises](#sec8)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Why not plain linear regression?

`converted` only takes the values `0` or `1`. Let's try fitting a plain linear
regression (Module 12) to it anyway, against `monthly_spend`, and see what happens for
high values of the predictor:

In [ ]:
from statsmodels.formula.api import ols

linear_model = ols("converted ~ monthly_spend", data=customers).fit()
print(linear_model.params)

extreme_prediction = linear_model.predict(pd.DataFrame({"monthly_spend": [2000.0]}))
print(f"Predicted 'probability' of conversion at spend=2000: {extreme_prediction.values[0]:.3f}")


> ⚠️ **A negative probability doesn't make sense**
>
> Plain linear regression has NO built-in constraint at all -- the straight line freely goes outside the `[0, 1]` range, giving nonsensical "probabilities" that are negative or greater than 1. We need a model that, by construction, only returns values in the `[0, 1]` range.

<a id="sec2"></a>
## 2. The logistic function and the logit

The **logistic function** (*sigmoid*) transforms any real number into a value in the
range `(0, 1)`:

```
p = 1 / (1 + e^(-z))
```

where `z = b0 + b1*x1 + b2*x2 + ...` (the exact same linear combination of predictors as
in linear regression!). The inverse of this function is the **logit** (log-odds):

```
logit(p) = ln(p / (1 - p)) = z
```

In other words: logistic regression models LINEARLY not the probability `p` itself
(which must lie in `[0, 1]`), but the *logit* of that probability (which can take any
real value).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-8, 8, 200)
p = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 5))
plt.plot(z, p)
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("z (linear combination of predictors)")
plt.ylabel("p (probability)")
plt.title("Logistic function (sigmoid)")
plt.show()


<a id="sec3"></a>
## 3. Fitting a logistic regression

Let's check whether `campaign_group` (A/B) affects `converted` -- exactly the question we
studied with the chi-square test in Module 9 and the confidence interval for proportions
in Module 6.

In [ ]:
from statsmodels.formula.api import logit

model = logit("converted ~ C(campaign_group)", data=customers).fit(disp=0)
print(model.summary())


> 📊 **Logistic regression with a single categorical predictor = a proportions test**
>
> When the only predictor is a two-level categorical variable, logistic regression and the chi-square/proportions test from Modules 6 and 9 answer the same question with a very similar result. The real power of logistic regression shows up when we want to include MULTIPLE predictors at once (section 4 below) -- something a plain chi-square test can't do.

<a id="sec4"></a>
## 4. Interpretation: the odds ratio

Logistic regression coefficients are on the LOGIT scale, so they can't be read directly
the way linear regression coefficients can. Instead we transform them with the
exponential function, getting an **odds ratio**:

In [ ]:
odds_ratios = np.exp(model.params)
print(odds_ratios)


An odds ratio `> 1` means higher odds of conversion, `< 1` -- lower, `= 1` -- no
effect. For `campaign_group[T.B]` the odds ratio is approximately `2` -- meaning the
**odds** (not the probability directly, but the ratio `p / (1-p)`) of conversion in
group B are about twice as high as in group A.

In [ ]:
conversion_rate = customers.groupby("campaign_group")["converted"].mean()
print(conversion_rate)

odds_a = conversion_rate["A"] / (1 - conversion_rate["A"])
odds_b = conversion_rate["B"] / (1 - conversion_rate["B"])
print(f"Odds A: {odds_a:.3f}, odds B: {odds_b:.3f}")
print(f"Odds ratio B/A: {odds_b / odds_a:.3f}")


> 💡 **An odds ratio is not the same as 'how many times higher the probability is'**
>
> This is one of the most common interpretation mistakes: an odds ratio of `2` does NOT mean the probability is twice as high (`0.118 * 2 = 0.236`, while the actual `p` in group B is `0.218` -- close, but not identical). An odds ratio speaks about the ratio of ODDS (`p/(1-p)`), not probabilities themselves. The difference is small when `p` is small, but grows substantially as `p` approaches `0.5` or higher.

<a id="sec5"></a>
## 5. Predicting probabilities

The `.predict()` method in logistic regression directly returns PROBABILITIES (already
passed through the logistic function), not the raw logit:

In [ ]:
new_customers = pd.DataFrame({"campaign_group": ["A", "B"]})
predicted_probabilities = model.predict(new_customers)
print(predicted_probabilities)


<a id="sec6"></a>
## 6. Decision threshold and confusion matrix

To turn a probability into a concrete decision (convert yes/no), we need a **decision
threshold** -- most commonly `0.5` by default. Let's see how well the model with a `0.5`
threshold classifies ALL of our customers:

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

customers["probability"] = model.predict(customers)
customers["predicted_class"] = (customers["probability"] >= 0.5).astype(int)

cm = confusion_matrix(customers["converted"], customers["predicted_class"])
print("Confusion matrix:")
print(cm)
print(f"Accuracy: {accuracy_score(customers['converted'], customers['predicted_class']):.3f}")
print(customers["predicted_class"].value_counts())


> ⚠️ **The accuracy paradox**
>
> See what happened? The model predicts class `0` (no conversion) for ABSOLUTELY EVERY customer -- because even in group B, the predicted probability (`~0.22`) doesn't exceed the `0.5` threshold. And yet accuracy comes out high (about 83%)! That's because the baseline conversion rate itself is LOW (about 17% of customers convert) -- a model that "always guesses no" gets most cases right without learning anything useful at all. With imbalanced classes, accuracy alone is misleading -- we need better metrics (section 7).

<a id="sec7"></a>
## 7. Precision and recall

**Precision** -- of the predicted conversions, how many were actually conversions?
**Recall** -- of the actual conversions, how many did the model catch? At a `0.5`
threshold, the model from section 6 never predicts class `1` at all, so both measures
are `0`. Let's lower the threshold so the model actually starts detecting something:

In [ ]:
from sklearn.metrics import precision_score, recall_score

for threshold in [0.5, 0.2, 0.15]:
    class_at_threshold = (customers["probability"] >= threshold).astype(int)
    precision = precision_score(customers["converted"], class_at_threshold, zero_division=0)
    recall = recall_score(customers["converted"], class_at_threshold, zero_division=0)
    accuracy = accuracy_score(customers["converted"], class_at_threshold)
    print(f"threshold={threshold}: precision={precision:.3f}, recall={recall:.3f}, accuracy={accuracy:.3f}")


> 📊 **Picking a threshold is a trade-off, not one 'correct' value**
>
> A lower threshold catches more actual conversions (higher recall), at the cost of more false alarms (lower precision) and lower overall accuracy. Which threshold is "better" depends entirely on the business context -- e.g. when the cost of missing a conversion is high (a marketing campaign) and the cost of a false alarm is low, it's worth lowering the threshold.

<a id="sec8"></a>
## 8. Summary and exercises

In this module we covered:
- why plain linear regression isn't suited to predicting a 0/1 outcome,
- the logistic function (sigmoid) and the logit as a linear combination of predictors,
- fitting a logistic regression with `statsmodels.formula.api.logit`,
- interpreting coefficients as odds ratios (`np.exp`),
- predicting probabilities with `.predict()`,
- the decision threshold, the confusion matrix, and the accuracy paradox with
  imbalanced classes,
- precision and recall as better metrics for imbalanced classes.

> 📝 **Exercise 1: Does age affect conversion?**
>
> Build the model `converted ~ C(campaign_group) + age`. Check the p-value of the `age` coefficient. Does age add real predictive value beyond what `campaign_group` already gives?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
model_with_age = logit("converted ~ C(campaign_group) + age", data=customers).fit(disp=0)
print(model_with_age.summary())
```

Age's p-value comes out well above 0.05 (not significant) -- as in every previous module, age is drawn in our data completely independently of conversion, so it adds no real predictive value.
</details>

> 📝 **Exercise 2: Interpreting an odds ratio in a sentence**
>
> Compute the odds ratio for `campaign_group[T.B]` from the model in section 3. Write one comment sentence that correctly interprets this number (remember the trap from section 4 -- an 'odds ratio' is not the same as a 'probability ratio').

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
odds_ratio_b = np.exp(model.params["C(campaign_group)[T.B]"])
print(f"Odds ratio B vs A: {odds_ratio_b:.3f}")
# The ODDS (not probability!) of conversion in group B are about 2 times higher
# than the odds of conversion in group A.
```
</details>

> 📝 **Exercise 3: Confusion matrix at a lower threshold**
>
> Build a confusion matrix at a threshold of `0.18` (close to the overall conversion rate). Compare the number of false alarms (customers predicted to convert who did not) with the `0.5` threshold from section 6.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
class_018 = (customers["probability"] >= 0.18).astype(int)
print(confusion_matrix(customers["converted"], class_018))
```
</details>

> 📝 **Exercise 4: Predicting for a new customer**
>
> Use the model from section 3 to predict the conversion probability for a new customer in group A and a new customer in group B at once, in a single DataFrame.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
new_ones = pd.DataFrame({"campaign_group": ["A", "B"]})
model.predict(new_ones)
```
</details>

> 🔥 **Challenge: does controlling for spend change the campaign group effect?**
>
> Build the model `converted ~ C(campaign_group) + monthly_spend`. Compare the odds ratio for `campaign_group[T.B]` with this model against the model from section 3 (without spend). Does adding spend as a control predictor meaningfully change the conclusion about the campaign group's effect -- unlike the Module 12 challenge for satisfaction?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
full_model = logit("converted ~ C(campaign_group) + monthly_spend", data=customers).fit(disp=0)
print(np.exp(full_model.params))
print(f"p-value for spend: {full_model.pvalues['monthly_spend']:.4f}")
```

The odds ratio for campaign_group[T.B] stays almost identical (about 2.05 vs about 2.08 without spend), and monthly_spend isn't significant at all (p clearly above 0.05). This contrasts with the Module 12 challenge: there, controlling for spend DID change the conclusion about campaign group's significance for satisfaction, because satisfaction genuinely depended on spend. Here, conversion in the data depends ONLY on campaign group, so adding spend changes nothing -- controlling for a variable only matters when that variable actually explains something.
</details>

## What's next?

So far we've tested individual hypotheses and built models on data that was already
collected. In **Module 14** we'll tackle the other side of an A/B test: how to DESIGN
one well from the start (sample size, statistical power) and how to avoid common
practical pitfalls (peeking at results early, multiple comparisons).